# NB06d — K-Means, PCA and Sentiment in Finance

**Role:** Optional enrichment  
**Manual section:** 2.3.2.a — Unsupervised learning; 2.3.2.d — Text and sentiment  
**Prerequisites:** NB06

---

## Purpose of this notebook

This optional enrichment notebook provides hands-on practice with K-Means clustering (including elbow method and segment profiling), PCA for dimensionality reduction, and keyword-based sentiment scoring on financial headlines.

NB06b introduced K-Means briefly; this notebook provides the full treatment.

**Datasets:** `dataset_D_unsupervised_extension.csv` + `dataset_G_finance_headlines_sentiment.csv`

---

## Section 1 — K-Means clustering

> NB06b introduces K-Means briefly; this notebook extends the analysis with the elbow method, profiling and a PCA combination.

### Intuition

K-Means groups observations into *k* clusters so that points within the same
cluster are as similar as possible (close together in feature space).

**How it works (simplified):**

1. Pick *k* random starting points (centroids).
2. Assign every observation to the nearest centroid.
3. Move each centroid to the mean of its assigned observations.
4. Repeat steps 2–3 until the assignments stop changing.

**Key requirements:**
- Features must be **numeric** (distances between categories are not meaningful).
- Features must be **scaled** — otherwise a variable with a large range (e.g.,
  salary in thousands) dominates the distance calculation.
- The number of clusters *k* must be **chosen by the analyst** — the algorithm
  does not decide for you.

### Demo with Dataset D

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Resolve data directory — works from the project root or notebooks/
_candidates = [Path("data/processed"), Path("../data/processed")]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/. "
        "Launch the notebook from the project root or the notebooks/ folder."
    )

df_d = pd.read_csv(DATA_DIR / "dataset_D_unsupervised_extension.csv")
print(f"Dataset D: {df_d.shape[0]:,} rows × {df_d.shape[1]} columns")
print(f"Columns: {list(df_d.columns)}")
df_d.describe().round(1)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Always scale before K-Means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_d)

print("Scaled feature matrix shape:", X_scaled.shape)

### Choosing k with the elbow method

The **elbow method** runs K-Means for several values of *k* and plots the total
within-cluster distance (inertia). We look for a "bend" where adding more
clusters gives diminishing returns.

In [ ]:
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(list(K_range), inertias, "o-", color="steelblue")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Inertia (within-cluster distance)")
ax.set_title("Elbow method — Dataset D")
plt.tight_layout()
plt.show()

> There is rarely a perfectly obvious elbow. Use it as guidance, not a strict
> rule. Domain knowledge should also inform the choice — does it make business
> sense to have 3 customer segments? Or 5?

### Fitting K-Means with k = 4

In [ ]:
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df_d["cluster"] = km.fit_predict(X_scaled)

print("Cluster sizes:")
print(df_d["cluster"].value_counts().sort_index())

In [ ]:
# Cluster profiles — mean of each feature per cluster
profile = df_d.groupby("cluster").mean(numeric_only=True).round(1)
profile

### Interpreting the clusters

Look at the cluster profiles and try to assign **business-meaningful names**.
For example:

- High balance + high salary + low complaints → "Premium low-risk"
- Many products + high digital usage → "Digitally engaged multi-product"
- Low tenure + high complaints → "At-risk new customers"

> **Remember:** these labels are interpretations you create — the algorithm only
> finds groups of similar observations. Good segmentation requires both
> statistical output **and** business judgment.

---

## Section 2 — Common mistakes with clustering

| Mistake | Why it matters |
|---------|---------------|
| **Forgetting to scale** | Salary (range ~100 000) would dominate over tenure (range ~10) |
| **Treating clusters as truth** | Clusters are a model output, not a discovered law of nature |
| **Ignoring the choice of features** | Different features → different clusters. The result depends on what you measure |
| **Over-interpreting small differences** | A cluster mean of 52 vs 54 for age is not a meaningful distinction |
| **Choosing *k* mechanically** | The elbow method is a guide; business context should have the final say |

> Clustering is a **starting point for analysis**, not an end product. Always
> validate clusters with domain experts and additional data if possible.

---

## Section 3 — PCA (Principal Component Analysis)

### Intuition

When you have many correlated features, some of the "information" is redundant.
PCA **creates new features** (called components) that are:

- ordered by how much variance they capture,
- uncorrelated with each other,
- linear combinations of the original features.

**Think of it as:** compressing a 8-dimensional spreadsheet into 2–3 summary
dimensions that capture most of the variation.

### When PCA is useful

| Use case | Benefit |
|----------|---------|
| Visualisation | Plot high-dimensional data in 2D |
| Noise reduction | Discard low-variance components that may be noise |
| Feature compression | Reduce the number of inputs to a model |
| Multicollinearity | Replace correlated features with uncorrelated components |

### Demo with Dataset D

In [ ]:
from sklearn.decomposition import PCA

# Fit PCA on the scaled features (without the cluster column)
X_for_pca = X_scaled  # already scaled from earlier

pca = PCA(random_state=42)
X_pca = pca.fit_transform(X_for_pca)

print(f"Original features: {X_for_pca.shape[1]}")
print(f"PCA components:    {pca.n_components_}")
print()
print("Explained variance ratio per component:")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.3f}  ({var*100:.1f} %)")

In [ ]:
# Cumulative explained variance
cumvar = np.cumsum(pca.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(range(1, len(cumvar)+1), pca.explained_variance_ratio_,
       color="steelblue", alpha=0.7, label="Individual")
ax.step(range(1, len(cumvar)+1), cumvar, where="mid",
        color="darkorange", linewidth=2, label="Cumulative")
ax.set_xlabel("Principal component")
ax.set_ylabel("Explained variance ratio")
ax.set_title("PCA — Explained variance per component (Dataset D)")
ax.legend()
ax.set_xticks(range(1, len(cumvar)+1))
plt.tight_layout()
plt.show()

print(f"First 2 components explain {cumvar[1]*100:.1f} % of total variance.")
print(f"First 3 components explain {cumvar[2]*100:.1f} % of total variance.")

> In practice, there is no fixed rule for how many components to keep.
> A common guideline is to keep enough to explain **80–90 %** of the variance,
> but the right choice depends on the problem.

---

## Section 4 — PCA + clustering combined

PCA gives us a 2D projection to **visualise** the clusters we found earlier.
This is often more informative than picking two raw features.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=df_d["cluster"], cmap="Set2", alpha=0.4, s=10)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("K-Means clusters in PCA space (Dataset D)")
plt.colorbar(scatter, ax=ax, label="Cluster")
plt.tight_layout()
plt.show()

> The 2D PCA projection does not show all the information — the clustering
> was done in 8 dimensions. But it gives a useful visual summary.
>
> If clusters overlap heavily in this plot, it may mean either:
> - the clusters are not very distinct in the data, or
> - the first two components do not capture the dimensions that separate them.
>
> **PCA and clustering are complementary tools**, not competing ones.

---

## Section 5 — Sentiment analysis: a lightweight example

### What is sentiment analysis?

Sentiment analysis tries to determine the **emotional tone** of a piece of text —
positive, negative or neutral.

In finance, it is applied to:
- news headlines,
- earnings call transcripts,
- analyst reports,
- social media posts about stocks.

### Dataset G — Finance headlines

Dataset G contains **144 real finance-related headlines** (mostly about GME from
early 2021), each with a pre-assigned sentiment label.

> **Important warning:** these sentiment labels are approximate teaching labels,
> not gold-standard annotations. Real sentiment labelling is noisy and
> context-dependent. Treat them as illustrative, not as ground truth.

In [ ]:
df_g = pd.read_csv(DATA_DIR / "dataset_G_finance_headlines_sentiment.csv")
print(f"Dataset G: {df_g.shape[0]} rows × {df_g.shape[1]} columns")
print()
print("Sentiment distribution:")
print(df_g["sentiment_label"].value_counts())
print()
df_g.head()

In [ ]:
# Sentiment breakdown with sample headlines
for label in df_g["sentiment_label"].unique():
    subset = df_g[df_g["sentiment_label"] == label]
    print(f"\n--- {label.upper()} ({len(subset)} headlines) ---")
    for h in subset["headline"].head(2).values:
        print(f"  • {h[:100]}")

### Simple keyword-based sentiment scoring

We build a **very simple** sentiment scorer using keyword lists. This is
intentionally basic — real NLP systems use much more sophisticated methods.
The goal here is to illustrate the *concept*, not to build a production system.

In [ ]:
# Very simple keyword lists (for illustration only)
positive_words = {"gain", "gains", "profit", "profits", "up", "rise", "rises",
                  "rally", "rallies", "bull", "bullish", "growth", "surge",
                  "buy", "hold", "moon", "rocket", "love", "win", "winning",
                  "strong", "strength", "good", "great", "amazing", "positive"}
negative_words = {"loss", "losses", "down", "drop", "drops", "fall", "falls",
                  "crash", "bear", "bearish", "decline", "sell", "selling",
                  "risk", "fear", "panic", "bad", "worst", "fail", "failure",
                  "weak", "weakness", "negative", "trouble", "plunge"}

def simple_sentiment(text):
    words = set(str(text).lower().split())
    pos = len(words & positive_words)
    neg = len(words & negative_words)
    if pos > neg:
        return "positive"
    elif neg > pos:
        return "negative"
    else:
        return "neutral"

df_g["keyword_sentiment"] = df_g["headline"].apply(simple_sentiment)

# Compare with the pre-assigned label
agreement = (df_g["keyword_sentiment"] == df_g["sentiment_label"]).mean()
print(f"Agreement between keyword method and pre-assigned labels: {agreement:.1%}")

> The low agreement rate shows why keyword-based methods are **limited**.
> They miss context, sarcasm, negation ("not good"), and domain-specific language.
> Modern NLP methods (transformers, fine-tuned models) are much better, but they
> require more infrastructure.

In [ ]:
# Aggregate sentiment by market tag
sentiment_summary = (
    df_g.groupby(["market_tag", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
)
print("Sentiment by market tag:")
print(sentiment_summary)

# Quick bar chart
sentiment_summary.plot(kind="bar", stacked=True, figsize=(7, 3.5),
                       color=["#d9534f", "#f0ad4e", "#5cb85c"])
plt.title("Headline sentiment by market tag (Dataset G)")
plt.ylabel("Count")
plt.xlabel("Market tag")
plt.xticks(rotation=0)
plt.legend(title="Sentiment")
plt.tight_layout()
plt.show()

---

## Section 6 — Interpretation and business caution

### Segmentation and sentiment can support — but not replace — business decisions

| Method | What it provides | What it does NOT provide |
|--------|-----------------|------------------------|
| **K-Means** | Groups of similar observations | A guaranteed "real" segmentation — clusters depend on features and *k* |
| **PCA** | A compressed view of the data | A causal explanation of why variables are correlated |
| **Sentiment** | A rough signal about tone | An accurate prediction of market direction |

### Key warnings

- **Clusters are not segments until validated by business experts.**
  The algorithm finds statistical groupings; only domain knowledge determines
  whether those groupings are useful.
- **PCA components are mathematical summaries, not stories.**
  Naming them ("the risk component") is interpretation, not fact.
- **Sentiment is noisy per individual headline.**
  It becomes more useful when aggregated over many texts and combined with
  other data sources.
- **None of these methods should be used in isolation for financial decisions.**
  They are inputs to a broader analytical process.

---

## Section 7 — Recap

### What to remember

| Method | One-line summary |
|--------|-----------------|
| **K-Means** | Groups observations into *k* clusters based on feature similarity — requires scaling and a choice of *k* |
| **PCA** | Compresses correlated features into ordered components that maximise explained variance |
| **Sentiment** | Estimates the emotional tone of text — noisy but potentially useful as an additional signal |

> **All three are tools in the broader data science toolbox.** They complement
> supervised models (NB04–06) by addressing questions where there is no clear
> target variable or where text data adds value.

---

### Self-practice

1. Re-run K-Means with *k* = 3 and *k* = 6. Compare the cluster profiles.
   Which number of clusters tells a more useful story?
2. Look at the cumulative explained variance chart. How many PCA components
   would you keep to capture at least 80 % of the variance?
3. Add two words to `positive_words` and two to `negative_words`. Does the
   agreement rate change? What does this tell you about keyword methods?
4. Pick one cluster from the K-Means output and write a 3-sentence
   "customer segment brief" that a marketing team could act on.
5. Explain in one sentence why PCA components are harder to interpret than
   original features.

---

*This is an optional enrichment notebook supporting manual sections 2.3.2.a and 2.3.2.d.*